In [ ]:
# --- Repository setup ---
import os
from pathlib import Path

# Find the repository root (the folder that contains data/graphs) and work from there,
# so every relative path below resolves no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'graphs').is_dir())
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)


# Phase 1 — Virtual Patient Backstory Generation (Hidden Context)
This notebook adds a `user_story` field to previously generated JSON cases.

Each story is a rich backstory that implicitly answers every YES/NO condition in the extracted path.


## Configuration and imports
Set the input/output roots and your OpenAI API key before running generation.


In [3]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

# Optional if your notebook environment does not already have the SDK:
# %pip install -q openai

PROJECT_ROOT = REPO_ROOT / 'data' / 'graphs'
INPUT_ROOT = REPO_ROOT / 'data' / 'cases' / 'reference_paths'
OUTPUT_ROOT = REPO_ROOT / 'data' / 'cases' / 'virtual_patients'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Faster, lower-cost model for story generation.
OPENAI_MODEL = 'gpt-5.5'

print('Input root :', INPUT_ROOT)
print('Output root:', OUTPUT_ROOT)
print('Model      :', OPENAI_MODEL)


Input root : /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple
Output root: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories
Model      : gpt-5.5


## Helpers
These helpers load cases, generate user stories, and save outputs into per-module folders.


In [4]:
def load_json(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8'))


def save_json(data: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    return path


def ensure_openai_client():
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY is not set. Set it before running generation.')
    try:
        from openai import OpenAI
    except ModuleNotFoundError as exc:
        raise RuntimeError('OpenAI SDK is not installed. In a notebook cell, run: %pip install openai') from exc
    return OpenAI()


def extract_conditions(case: dict[str, Any]) -> list[dict[str, Any]]:
    """Return the YES/NO conditions implied by the case path.

    Preferred (new) format: node_sequence with per-node condition_met (meaning: the edge
    condition taken to reach the next node).

    Back-compat (old) format: relationship_sequence.
    """

    def _extract_from_node_sequence(node_sequence: Any) -> list[dict[str, Any]]:
        if not isinstance(node_sequence, list):
            return []
        out: list[dict[str, Any]] = []
        for i, node in enumerate(node_sequence[:-1]):
            if not isinstance(node, dict):
                continue
            nxt = node_sequence[i + 1]
            if not isinstance(nxt, dict):
                continue
            condition = node.get('condition_met')
            # The story only needs to implicitly answer YES/NO decisions.
            if condition in ('YES', 'NO'):
                out.append({'from': node.get('id'), 'to': nxt.get('id'), 'condition': condition})
        return out

    node_conditions = _extract_from_node_sequence(case.get('node_sequence'))
    if node_conditions:
        return node_conditions

    # Fallback for older case JSON files.
    conditions: list[dict[str, Any]] = []
    for rel in case.get('relationship_sequence', []):
        if not isinstance(rel, dict):
            continue
        condition = rel.get('condition')
        if condition in ('YES', 'NO'):
            conditions.append({'from': rel.get('from'), 'to': rel.get('to'), 'condition': condition})
    return conditions



def build_story_prompt(case: dict[str, Any]) -> dict[str, Any]:
    return {
        'task': 'Generate a rich, concise short virtual patient backstory with a unique persona.',
       
        'requirements': [
    'The persona must be Bangladeshi (name, setting, and life context should fit Bangladesh).',
    'Create a unique persona for each case; do not reuse the same name, age, occupation, or location across cases.',
    'The backstory must implicitly answer every YES/NO condition in the provided path or node_sequence.',
    'Both YES and NO conditions must be represented in the story.',
    'For NO conditions, include realistic evidence that the symptom, behavior, risk factor, or event is absent; do not simply omit it.',
    'The story should contain enough information for an assessor to infer the correct answer without guessing.',
    'Do not list answers explicitly as a checklist; weave them into the narrative.',
    'Stay consistent with the path sequence and clinical context.',
    'Keep the story realistic and medically coherent.',
    
    ],

        'node_sequence': case.get('node_sequence', []),
        'conditions': extract_conditions(case),
        'output_contract': {
            'format': 'text_only',
            'field': 'user_story',
        },
    }


def generate_user_story(case: dict[str, Any], *, model: str = OPENAI_MODEL) -> str:
    client = ensure_openai_client()
    prompt_payload = build_story_prompt(case)
    response = client.responses.create(
        model=model,
        input=[
            {
                'role': 'developer',
                'content': (
                    'You generate a virtual patient backstory (hidden context) for clinical decision paths. '
                    'Return only the story text. Do not add headers, bullet points, or JSON.'
                ),
            },
            {
                'role': 'user',
                'content': json.dumps(prompt_payload, ensure_ascii=False),
            },
        ],
    )
    story = (response.output_text or '').strip()
    if not story:
        raise ValueError('OpenAI returned empty story text')
    return story


def enrich_cases_with_user_stories(payload: dict[str, Any]) -> dict[str, Any]:
    cases = payload.get('cases', [])
    for case in cases:
        case['user_story'] = generate_user_story(case)
    return payload


def module_folder_name(payload: dict[str, Any]) -> str:
    cases = payload.get('cases', [])
    if cases:
        return str(cases[0].get('source_module', 'UNKNOWN'))
    return 'UNKNOWN'


## Single file processing
Use this cell when you want to enrich one JSON file at a time.


In [5]:
# Example: replace with your specific JSON file path
input_path = INPUT_ROOT / 'DEP' / 'DEP_single_hop_single_hop_positive_1.json'

payload = load_json(input_path)
payload = enrich_cases_with_user_stories(payload)

module_folder = module_folder_name(payload)
output_path = OUTPUT_ROOT / module_folder / input_path.name
saved = save_json(payload, output_path)
print('Saved:', saved)


KeyboardInterrupt: 

In [7]:
input_path = INPUT_ROOT / 'DEP' / 'DEP_single_hop_single_hop_negative_1.json'

payload = load_json(input_path)
payload = enrich_cases_with_user_stories(payload)

module_folder = module_folder_name(payload)
output_path = OUTPUT_ROOT / module_folder / input_path.name
saved = save_json(payload, output_path)
print('Saved:', saved)

Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_single_hop_single_hop_negative_1.json


In [ ]:
input_path = INPUT_ROOT / 'DEP' / 'DEP_to_SUI_multi_hop_multi_hop_positive_1.json'

payload = load_json(input_path)
payload = enrich_cases_with_user_stories(payload)

module_folder = module_folder_name(payload)
output_path = OUTPUT_ROOT / module_folder / input_path.name
saved = save_json(payload, output_path)
print('Saved:', saved)

In [4]:
input_path = INPUT_ROOT / 'DEP' / 'DEP_to_CMH_multi_hop_multi_hop_positive_1.json'

payload = load_json(input_path)
payload = enrich_cases_with_user_stories(payload)

module_folder = module_folder_name(payload)
output_path = OUTPUT_ROOT / module_folder / input_path.name
saved = save_json(payload, output_path)
print('Saved:', saved)

Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_to_CMH_multi_hop_multi_hop_positive_1.json


In [8]:
input_path = INPUT_ROOT / 'DEP' / 'DEP_to_PSY_multi_hop_multi_hop_positive_1.json'

payload = load_json(input_path)
payload = enrich_cases_with_user_stories(payload)

module_folder = module_folder_name(payload)
output_path = OUTPUT_ROOT / module_folder / input_path.name
saved = save_json(payload, output_path)
print('Saved:', saved)

Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_to_PSY_multi_hop_multi_hop_positive_1.json


In [9]:
input_path = INPUT_ROOT / 'DEP' / 'DEP_to_OTH_multi_hop_multi_hop_positive_1.json'

payload = load_json(input_path)
payload = enrich_cases_with_user_stories(payload)

module_folder = module_folder_name(payload)
output_path = OUTPUT_ROOT / module_folder / input_path.name
saved = save_json(payload, output_path)
print('Saved:', saved)

Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/DEP/DEP_to_OTH_multi_hop_multi_hop_positive_1.json


In [6]:
input_files = sorted(INPUT_ROOT.rglob('*.json'))
print('Found', len(input_files), 'files')


Found 22 files


In [9]:
temp = input_files[0:3]

In [10]:
temp1 = input_files[9:]

In [11]:
final = temp + temp1

In [12]:
final

[PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_single_hop_single_hop_negative_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_single_hop_single_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_to_SUI_multi_hop_multi_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/EPI/EPI_single_hop_single_hop_negative_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/EPI/EPI_single_hop_single_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/EPI/EPI_to_SUI_multi_hop_multi_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/PSY/PSY_single_hop_single_hop_negative_1.json'),
 Po

In [8]:
input_files

[PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_single_hop_single_hop_negative_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_single_hop_single_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_to_SUI_multi_hop_multi_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_single_hop_single_hop_negative_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_single_hop_single_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_to_CMH_multi_hop_multi_hop_positive_1.json'),
 PosixPath('/Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_to_OTH_multi_hop_multi_hop_positive_1.json')

## Batch processing
Enrich all JSON files from the Phase 1 generation outputs and save them into per-module folders.


In [13]:


for input_path in final:
    payload = load_json(input_path)
    payload = enrich_cases_with_user_stories(payload)
    module_folder = module_folder_name(payload)
    output_path = OUTPUT_ROOT / module_folder / input_path.name
    saved = save_json(payload, output_path)
    print('Saved:', saved)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/CMH/CMH_single_hop_single_hop_negative_1.json
Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/CMH/CMH_single_hop_single_hop_positive_1.json
Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/CMH/CMH_to_SUI_multi_hop_multi_hop_positive_1.json
Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/EPI/EPI_single_hop_single_hop_negative_1.json
Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/EPI/EPI_single_hop_single_hop_positive_1.json
Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/EPI/EPI_to_SUI_multi_hop_multi_hop_positive_1.json
Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_userstories/PSY/PSY_single_hop_single_hop_negative_1.json
Saved: /Users/rakib/Downloads/mental-health-r